# U-Net Baseline — Forest Semantic Segmentation
**Paper:** Improved Architecture for Semantic Segmentation of Vegetation Cover using Deep Neural Networks  
**Model:** U-Net (Ronneberger et al., 2015) with BatchNorm — binary forest / non-forest segmentation  
**Dataset:** Forest Segmented (5108 × 256×256 satellite image-mask pairs)

---
### Colab setup instructions
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload the `Forest Segmented` dataset folder to your Google Drive under `My Drive/DNN-Project/data/`  
   Expected structure:  
   ```
   My Drive/DNN-Project/data/Forest Segmented/Forest Segmented/
       images/
       masks/
       meta_data.csv
   ```
3. Run all cells in order.

## 1 · Install dependencies

In [ ]:
# All packages are pre-installed on Colab — just verify
import subprocess, sys
pkgs = ['torch', 'torchvision', 'numpy', 'Pillow', 'matplotlib', 'tqdm', 'pandas']
for p in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'show', p],
                       capture_output=True, text=True)
    ver = [l for l in r.stdout.splitlines() if l.startswith('Version')]
    print(p, '-', ver[0] if ver else 'NOT FOUND')

## 2 · Mount Google Drive & configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── adjust DRIVE_BASE if you placed the dataset in a different folder ──────────
DRIVE_BASE     = '/content/drive/MyDrive/DNN-Project'
DATA_DIR       = os.path.join(DRIVE_BASE, 'data', 'Forest Segmented', 'Forest Segmented')
IMG_DIR        = os.path.join(DATA_DIR, 'images')
MASK_DIR       = os.path.join(DATA_DIR, 'masks')
CSV_PATH       = os.path.join(DATA_DIR, 'meta_data.csv')

# Outputs go to fast local Colab storage (not Drive) during training
CHECKPOINT_DIR = '/content/checkpoints'
RESULTS_DIR    = '/content/results'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)

# Verify dataset is reachable
import glob
imgs  = glob.glob(os.path.join(IMG_DIR,  '*.jpg')) or glob.glob(os.path.join(IMG_DIR,  '*.png'))
masks = glob.glob(os.path.join(MASK_DIR, '*.jpg')) or glob.glob(os.path.join(MASK_DIR, '*.png'))
print(f'Images : {len(imgs)}  |  Masks : {len(masks)}')
assert len(imgs) > 0, f'No images found in {IMG_DIR}'
assert len(masks) > 0, f'No masks found in {MASK_DIR}'
if len(imgs) != len(masks):
    print(f'[info] Count mismatch ({len(imgs)} imgs vs {len(masks)} masks) — '
          f'the dataset loader will use only matched CSV pairs, this is fine.')
print('Dataset OK — continuing')

## 3 · Configuration

In [ ]:
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE       = 256
MASK_THRESHOLD = 127     # binarise JPEG-compressed masks
TRAIN_RATIO    = 0.80
VAL_RATIO      = 0.10
RANDOM_SEED    = 42

IN_CHANNELS    = 3
OUT_CHANNELS   = 1
FEATURES       = [64, 128, 256, 512]

BATCH_SIZE     = 16      # raise to 32 on A100 for faster training
NUM_EPOCHS     = 50
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
NUM_WORKERS    = 2       # Colab supports multiprocessing

BCE_WEIGHT     = 0.5
DICE_WEIGHT    = 0.5
LR_ETA_MIN     = 1e-6

BEST_CKPT = os.path.join(CHECKPOINT_DIR, 'unet_baseline_best.pth')
LAST_CKPT = os.path.join(CHECKPOINT_DIR, 'unet_baseline_last.pth')

## 4 · Dataset & DataLoaders

In [ ]:
import random
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

_MEAN = [0.485, 0.456, 0.406]   # ImageNet statistics
_STD  = [0.229, 0.224, 0.225]


class ForestDataset(Dataset):
    """Binary forest segmentation dataset.

    Returns (image, mask) where:
      image : FloatTensor (3, H, W) — ImageNet-normalised
      mask  : FloatTensor (1, H, W) — 0=background, 1=forest
    """

    def __init__(self, df, img_dir, mask_dir,
                 img_size=256, augment=False, mask_threshold=127):
        self.df             = df.reset_index(drop=True)
        self.img_dir        = img_dir
        self.mask_dir       = mask_dir
        self.augment        = augment
        self.mask_threshold = mask_threshold

        self.img_tf = T.Compose([
            T.Resize((img_size, img_size),
                     interpolation=T.InterpolationMode.BILINEAR),
            T.ToTensor(),
            T.Normalize(mean=_MEAN, std=_STD),
        ])
        self.mask_resize = T.Resize(
            (img_size, img_size),
            interpolation=T.InterpolationMode.NEAREST
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(os.path.join(self.img_dir,  row['image'])).convert('RGB')
        mask = Image.open(os.path.join(self.mask_dir, row['mask'])).convert('L')

        if self.augment:
            if random.random() > 0.5:
                img, mask = TF.hflip(img), TF.hflip(mask)
            if random.random() > 0.5:
                img, mask = TF.vflip(img), TF.vflip(mask)
            angle = random.choice([0, 90, 180, 270])
            if angle:
                img  = TF.rotate(img,  angle)
                mask = TF.rotate(mask, angle)

        img_t  = self.img_tf(img)
        mask   = self.mask_resize(mask)
        mask_t = torch.from_numpy(
            (np.array(mask, dtype=np.float32) > self.mask_threshold
             ).astype(np.float32)
        ).unsqueeze(0)

        return img_t, mask_t


def make_splits(csv_path, train_ratio=0.8, val_ratio=0.1, seed=42):
    df = pd.read_csv(csv_path).sample(frac=1, random_state=seed).reset_index(drop=True)

    # Filter to only rows where BOTH image and mask files exist on disk
    exists = df.apply(
        lambda r: os.path.exists(os.path.join(IMG_DIR,  r['image'])) and
                  os.path.exists(os.path.join(MASK_DIR, r['mask'])),
        axis=1
    )
    full_len = len(df)
    df = df[exists].reset_index(drop=True)
    if len(df) < full_len:
        print(f'[info] {full_len - len(df)} CSV rows skipped (files not on Drive)')
    print(f'[info] Using {len(df)} valid image-mask pairs')

    n        = len(df)
    n_train  = int(n * train_ratio)
    n_val    = int(n * val_ratio)
    return df.iloc[:n_train], df.iloc[n_train:n_train+n_val], df.iloc[n_train+n_val:]


def get_loaders():
    train_df, val_df, test_df = make_splits(
        CSV_PATH, TRAIN_RATIO, VAL_RATIO, RANDOM_SEED
    )
    kw = dict(img_size=IMG_SIZE, mask_threshold=MASK_THRESHOLD)
    train_ds = ForestDataset(train_df, IMG_DIR, MASK_DIR, augment=True,  **kw)
    val_ds   = ForestDataset(val_df,   IMG_DIR, MASK_DIR, augment=False, **kw)
    test_ds  = ForestDataset(test_df,  IMG_DIR, MASK_DIR, augment=False, **kw)

    loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  **loader_kw)
    val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, **loader_kw)
    test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, **loader_kw)

    print(f'Train: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}')
    return train_loader, val_loader, test_loader


# Quick sanity check
train_loader, val_loader, test_loader = get_loaders()
imgs_b, masks_b = next(iter(train_loader))
print(f'Image batch : {imgs_b.shape}  {imgs_b.dtype}')
print(f'Mask  batch : {masks_b.shape}  unique={masks_b.unique().tolist()}')

## 5 · U-Net Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    """(Conv2d → BN → ReLU) × 2"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNet(nn.Module):
    """
    Vanilla U-Net with BatchNorm.
    Encoder  : 4 × DoubleConv + MaxPool  (64 → 128 → 256 → 512)
    Bottleneck: DoubleConv (512 → 1024)
    Decoder  : 4 × ConvTranspose2d + skip-concat + DoubleConv
    Head     : 1×1 Conv → raw logit
    """
    def __init__(self, in_channels=3, out_channels=1,
                 features=None):
        super().__init__()
        features = features or [64, 128, 256, 512]
        self.pool       = nn.MaxPool2d(2, 2)
        self.encoder    = nn.ModuleList()
        self.decoder_up = nn.ModuleList()
        self.decoder_dc = nn.ModuleList()

        ch = in_channels
        for f in features:
            self.encoder.append(DoubleConv(ch, f)); ch = f

        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        for f in reversed(features):
            self.decoder_up.append(nn.ConvTranspose2d(f*2, f, 2, 2))
            self.decoder_dc.append(DoubleConv(f*2, f))

        self.head = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for enc in self.encoder:
            x = enc(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        skips.reverse()
        for up, dc, skip in zip(self.decoder_up, self.decoder_dc, skips):
            x = up(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, skip.shape[2:], mode='bilinear',
                                  align_corners=False)
            x = dc(torch.cat([skip, x], dim=1))
        return self.head(x)


model = UNet(IN_CHANNELS, OUT_CHANNELS, FEATURES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'U-Net baseline  |  trainable parameters: {n_params:,}')

# Forward-pass shape check
with torch.no_grad():
    dummy = torch.randn(2, 3, 256, 256).to(DEVICE)
    out   = model(dummy)
    print(f'Output shape    : {out.shape}')   # expect (2, 1, 256, 256)

## 6 · Loss Function & Metrics

In [ ]:
# ── Loss ──────────────────────────────────────────────────────────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, logits, targets):
        p = torch.sigmoid(logits).view(-1)
        t = targets.view(-1)
        return 1.0 - (2*p*t + self.smooth).sum() / (p.sum() + t.sum() + self.smooth)


class BCEDiceLoss(nn.Module):
    """0.5 × BCE  +  0.5 × Dice  (both computed on raw logits)"""
    def __init__(self, bce_w=0.5, dice_w=0.5):
        super().__init__()
        self.bce_w = bce_w; self.dice_w = dice_w
        self.bce   = nn.BCEWithLogitsLoss()
        self.dice  = DiceLoss()
    def forward(self, logits, targets):
        return self.bce_w*self.bce(logits, targets) + self.dice_w*self.dice(logits, targets)


# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(logits, targets, thr=0.5):
    """Return dict: pixel_acc, precision, recall, f1, iou, dice"""
    p = (torch.sigmoid(logits) > thr).float()
    t = targets.float()
    eps = 1e-6
    TP = (p * t).sum()
    TN = ((1-p)*(1-t)).sum()
    FP = (p*(1-t)).sum()
    FN = ((1-p)*t).sum()
    prec  = TP / (TP + FP + eps)
    rec   = TP / (TP + FN + eps)
    return {
        'pixel_acc': ((TP+TN)/(TP+TN+FP+FN+eps)).item(),
        'precision': prec.item(),
        'recall'   : rec.item(),
        'f1'       : (2*prec*rec/(prec+rec+eps)).item(),
        'iou'      : (TP/(TP+FP+FN+eps)).item(),
        'dice'     : (2*TP/(2*TP+FP+FN+eps)).item(),
    }


def avg_metrics(ml):
    return {k: sum(m[k] for m in ml)/len(ml) for k in ml[0]}


criterion = BCEDiceLoss(BCE_WEIGHT, DICE_WEIGHT)
print('Loss and metric functions defined.')

## 7 · Training

In [ ]:
import csv, time
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=LR_ETA_MIN)


def run_epoch(model, loader, phase):
    is_train = (phase == 'train')
    model.train() if is_train else model.eval()
    total_loss, ml = 0.0, []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, masks)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()
            ml.append(compute_metrics(logits.detach(), masks))
    return total_loss / len(loader), avg_metrics(ml)


# ── Training loop ─────────────────────────────────────────────────────────────
log_path  = os.path.join(RESULTS_DIR, 'training_log.csv')
fields    = ['epoch','lr','train_loss','train_iou','train_dice','train_pixel_acc',
             'val_loss','val_iou','val_dice','val_pixel_acc']
log_f     = open(log_path, 'w', newline='')
writer    = csv.DictWriter(log_f, fieldnames=fields)
writer.writeheader()

best_iou  = 0.0
history   = {f: [] for f in ('train_loss','val_loss','train_iou','val_iou',
                              'train_dice','val_dice')}

epoch_bar = tqdm(range(1, NUM_EPOCHS+1), desc='Training', unit='epoch')
for epoch in epoch_bar:
    t0 = time.time()
    tl, tm = run_epoch(model, train_loader, 'train')
    vl, vm = run_epoch(model, val_loader,   'val')
    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]

    for k in history:
        src, met = k.split('_', 1)
        history[k].append(tm[met] if src=='train' else vm[met])

    row = dict(epoch=epoch, lr=f'{lr_now:.6f}',
               train_loss=f'{tl:.6f}', train_iou=f'{tm["iou"]:.6f}',
               train_dice=f'{tm["dice"]:.6f}', train_pixel_acc=f'{tm["pixel_acc"]:.6f}',
               val_loss=f'{vl:.6f}',   val_iou=f'{vm["iou"]:.6f}',
               val_dice=f'{vm["dice"]:.6f}',   val_pixel_acc=f'{vm["pixel_acc"]:.6f}')
    writer.writerow(row); log_f.flush()

    ckpt = dict(epoch=epoch, model_state=model.state_dict(),
                optimizer_state=optimizer.state_dict(),
                val_iou=vm['iou'], val_dice=vm['dice'])
    torch.save(ckpt, LAST_CKPT)
    if vm['iou'] > best_iou:
        best_iou = vm['iou']
        torch.save(ckpt, BEST_CKPT)

    epoch_bar.set_postfix({
        'tr_loss': f'{tl:.4f}', 'tr_iou': f'{tm["iou"]:.4f}',
        'vl_loss': f'{vl:.4f}', 'vl_iou': f'{vm["iou"]:.4f}',
        'time':    f'{time.time()-t0:.1f}s'
    })

log_f.close()
print(f'\nTraining complete.  Best val IoU = {best_iou:.4f}')
print(f'Best checkpoint → {BEST_CKPT}')

## 8 · Learning Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, NUM_EPOCHS + 1))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'],   label='Val')
axes[0].set(title='BCE + Dice Loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, history['train_iou'], label='Train')
axes[1].plot(epochs, history['val_iou'],   label='Val')
axes[1].set(title='IoU (Jaccard Index)', xlabel='Epoch', ylabel='IoU')
axes[1].legend(); axes[1].grid(True)

axes[2].plot(epochs, history['train_dice'], label='Train')
axes[2].plot(epochs, history['val_dice'],   label='Val')
axes[2].set(title='Dice Coefficient', xlabel='Epoch', ylabel='Dice')
axes[2].legend(); axes[2].grid(True)

fig.suptitle('U-Net Baseline — Training Curves', fontsize=13)
fig.tight_layout()
curve_path = os.path.join(RESULTS_DIR, 'training_curves.png')
fig.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {curve_path}')

## 9 · Test Set Evaluation

In [ ]:
# Load best checkpoint
ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded epoch {ckpt["epoch"]}  (val IoU={ckpt["val_iou"]:.4f})')

model.eval()
test_loss, test_ml = 0.0, []
sample_batch = None

with torch.no_grad():
    for i, (imgs, masks) in enumerate(tqdm(test_loader, desc='Testing', leave=True)):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        logits = model(imgs)
        test_loss += criterion(logits, masks).item()
        test_ml.append(compute_metrics(logits, masks))
        if sample_batch is None:
            sample_batch = (imgs.cpu(), masks.cpu(), logits.cpu())

test_loss /= len(test_loader)
m = avg_metrics(test_ml)

print('\n' + '='*50)
print('U-Net Baseline  —  Test Set Results')
print('='*50)
print(f'Loss (BCE+Dice) : {test_loss:.6f}')
print(f'IoU (Jaccard)   : {m["iou"]:.4f}')
print(f'Dice Coefficient: {m["dice"]:.4f}')
print(f'Pixel Accuracy  : {m["pixel_acc"]:.4f}')
print(f'Precision       : {m["precision"]:.4f}')
print(f'Recall          : {m["recall"]:.4f}')
print(f'F1 Score        : {m["f1"]:.4f}')
print('='*50)

# Save metrics to file
with open(os.path.join(RESULTS_DIR, 'test_metrics.txt'), 'w') as f:
    f.write(f'Loss            : {test_loss:.6f}\n')
    for k, v in m.items():
        f.write(f'{k:16s}: {v:.4f}\n')

## 10 · Qualitative Predictions

In [ ]:
def unnorm(t):
    """Inverse ImageNet normalisation → uint8 HxWx3 array."""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = t.numpy().transpose(1, 2, 0) * std + mean
    return np.clip(img * 255, 0, 255).astype(np.uint8)


imgs_cpu, masks_cpu, logits_cpu = sample_batch
preds = (torch.sigmoid(logits_cpu) > 0.5).float()
n     = min(8, imgs_cpu.shape[0])

fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
if n == 1: axes = axes[np.newaxis, :]

for j, title in enumerate(['Satellite Image', 'Ground Truth', 'U-Net Prediction']):
    axes[0, j].set_title(title, fontsize=10, fontweight='bold')

for i in range(n):
    axes[i, 0].imshow(unnorm(imgs_cpu[i]))
    axes[i, 1].imshow(masks_cpu[i, 0].numpy(), cmap='Greens', vmin=0, vmax=1)
    axes[i, 2].imshow(preds[i, 0].numpy(),     cmap='Greens', vmin=0, vmax=1)
    for j in range(3): axes[i, j].axis('off')

fig.suptitle('U-Net Baseline — Test Set Predictions', fontsize=12)
fig.tight_layout()
pred_path = os.path.join(RESULTS_DIR, 'test_predictions.png')
fig.savefig(pred_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {pred_path}')

## 11 · Save Results to Google Drive

In [ ]:
import shutil

# Copy all outputs from fast Colab local storage back to Google Drive
SAVE_DIR = os.path.join(DRIVE_BASE, 'outputs', 'unet_baseline')
os.makedirs(SAVE_DIR, exist_ok=True)

for src in [BEST_CKPT, LAST_CKPT,
            os.path.join(RESULTS_DIR, 'training_log.csv'),
            os.path.join(RESULTS_DIR, 'training_curves.png'),
            os.path.join(RESULTS_DIR, 'test_metrics.txt'),
            os.path.join(RESULTS_DIR, 'test_predictions.png')]:
    if os.path.exists(src):
        dst = os.path.join(SAVE_DIR, os.path.basename(src))
        shutil.copy2(src, dst)
        print(f'Saved  {os.path.basename(src):35s} → Drive')

print(f'\nAll outputs at:\n{SAVE_DIR}')